In [ ]:
# Shared paths: configure raw data once in config.local.toml at the repo root.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_project = next(
    (p for p in (_start, *_start.parents) if (p / "scripts" / "project_paths.py").is_file()),
    None,
)
if _project is None:
    raise RuntimeError("Start the notebook kernel in the repository or a subdirectory.")
_scripts = str(_project / "scripts")
if _scripts not in sys.path:
    sys.path.insert(0, _scripts)
from project_paths import PROJECT_ROOT, MIMIC_DATA_DIR, PROCESSED_DIR, REPORTS_DIR, mimic_csv


In [1]:
import pandas as pd
from pathlib import Path

processed_path = PROCESSED_DIR

train_data = pd.read_parquet(processed_path / "ml_train.parquet")
test_data = pd.read_parquet(processed_path / "ml_test.parquet")

X_train = train_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_train = train_data["HOSPITAL_EXPIRE_FLAG"]
subject_id_train = train_data["SUBJECT_ID"]

X_test = test_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_test = test_data["HOSPITAL_EXPIRE_FLAG"]

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]

numeric_cols = X_train.columns.difference(categorical_cols).tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ]
)

In [3]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

knn_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", knn_model)
    ]
)

setup = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

knn_results = cross_validate(
    knn_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", knn_results["test_roc_auc"].mean())
print("Precision:", knn_results["test_precision"].mean())
print("Recall:", knn_results["test_recall"].mean())
print("F1:", knn_results["test_f1"].mean())

ROC-AUC: 0.7091590759663919
Precision: 0.6137624370165354
Recall: 0.12919248688408602
F1: 0.2133674159043772


In [4]:
k_values = [3, 5, 7, 11, 15]

for k in k_values:
    knn_model = KNeighborsClassifier(
        n_neighbors=k
    )

    knn_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", knn_model)
        ]
    )

    knn_results = cross_validate(
        knn_pipeline,
        X_train,
        y_train,
        cv=setup,
        groups=subject_id_train,
        scoring=["roc_auc", "precision", "recall", "f1"]
    )

    print(f"K = {k}")
    print("ROC-AUC:", knn_results["test_roc_auc"].mean())
    print("Precision:", knn_results["test_precision"].mean())
    print("Recall:", knn_results["test_recall"].mean())
    print("F1:", knn_results["test_f1"].mean())
    print()

K = 3
ROC-AUC: 0.6673136709419384
Precision: 0.522708472088739
Recall: 0.16709313967614037
F1: 0.2531109222318987

K = 5
ROC-AUC: 0.7091590759663919
Precision: 0.6137624370165354
Recall: 0.12919248688408602
F1: 0.2133674159043772

K = 7
ROC-AUC: 0.7359815491563457
Precision: 0.6651515065661845
Recall: 0.10492837976745117
F1: 0.1810926269332796

K = 11
ROC-AUC: 0.7672152696835013
Precision: 0.7186984004127968
Recall: 0.07996662617309
F1: 0.14375082093410124

K = 15
ROC-AUC: 0.7821043572811052
Precision: 0.7661304787166856
Recall: 0.07187867946441681
F1: 0.1311396551499202



In [5]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate

smote_knn_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", KNeighborsClassifier(n_neighbors=3))
    ]
)

smote_knn_results = cross_validate(
    smote_knn_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", smote_knn_results["test_roc_auc"].mean())
print("Precision:", smote_knn_results["test_precision"].mean())
print("Recall:", smote_knn_results["test_recall"].mean())
print("F1:", smote_knn_results["test_f1"].mean())

ROC-AUC: 0.7391462259925051
Precision: 0.23554328911388844
Recall: 0.707882630925523
F1: 0.3534616570362564


In [6]:
k_values = [3, 5, 7]

for k in k_values:
    smote_knn_pipeline = ImbPipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("model", KNeighborsClassifier(n_neighbors=k))
        ]
    )

    smote_knn_results = cross_validate(
        smote_knn_pipeline,
        X_train,
        y_train,
        cv=setup,
        groups=subject_id_train,
        scoring=["roc_auc", "precision", "recall", "f1"]
    )

    print(f"K = {k}")
    print("ROC-AUC:", smote_knn_results["test_roc_auc"].mean())
    print("Precision:", smote_knn_results["test_precision"].mean())
    print("Recall:", smote_knn_results["test_recall"].mean())
    print("F1:", smote_knn_results["test_f1"].mean())
    print()

K = 3
ROC-AUC: 0.7391462259925051
Precision: 0.23554328911388844
Recall: 0.707882630925523
F1: 0.3534616570362564

K = 5
ROC-AUC: 0.7583589342156788
Precision: 0.2299160259730127
Recall: 0.7543341921531457
F1: 0.35240504877335693

K = 7
ROC-AUC: 0.7691184246252847
Precision: 0.22467615644292155
Recall: 0.7820638374561134
F1: 0.34906055917776796



## K-Nearest Neighbors (KNN)

KNN was evaluated as a distance-based classification model for ICU mortality prediction.

- Numerical features were processed using median imputation followed by `StandardScaler`.
- Categorical features were transformed using `OneHotEncoder`.
- Preprocessing and KNN were combined in a single pipeline.
- Model performance was evaluated with 5-fold `StratifiedGroupKFold`, keeping ICU stays from the same patient in the same fold.
- Different `n_neighbors` values were tested.
- Without class balancing, KNN showed low recall. The best recall among the tested standard KNN models was obtained with `K=3`:
  - ROC-AUC: **0.67**
  - Precision: **0.52**
  - Recall: **0.17**
  - F1: **0.25**
- Because mortality is the minority class, SMOTE was then applied inside the cross-validation pipeline.
- SMOTE substantially increased recall but reduced precision.
- With SMOTE:
  - `K=3`: Precision **0.24**, Recall **0.71**, F1 **0.35**
  - `K=5`: Precision **0.23**, Recall **0.75**, F1 **0.35**
  - `K=7`: Precision **0.22**, Recall **0.78**, F1 **0.35**
- `K=3 + SMOTE` was retained as the most balanced KNN candidate because it provided the highest precision and F1 among the SMOTE-based KNN models while maintaining high recall.
- Overall, KNN showed a weaker precision-recall balance than Logistic Regression for this dataset.